# Silver — Central Bank Indicators

Extracts policy rate and CPI from Bronze, fixes CPI date format, and writes a clean indicators table.

**Source:** `bronze.central_bank_raw`  
**Output:** `silver.central_bank_indicators` (Delta table)  
**Metrics:** `policy_rate` (daily), `cpi` (monthly)  

In [ ]:
df = spark.read.format("delta").table("bronze.central_bank_raw")

print(f"Bronze rows: {df.count()}")
df.groupBy("series_name").count().show(truncate=False)

In [ ]:
df_silver = spark.sql("""
    SELECT
        TO_DATE(date) AS date,
        'policy_rate' AS metric,
        series_name,
        ROUND(value, 4) AS value,
        CURRENT_TIMESTAMP() AS processed_at
    FROM bronze.central_bank_raw
    WHERE series_name = 'Meginvextir (vextir á 7 daga bundnum innlánum)'

    UNION ALL

    SELECT
        TO_DATE(date) AS date,
        'cpi' AS metric,
        series_name,
        ROUND(value, 4) AS value,
        CURRENT_TIMESTAMP() AS processed_at
    FROM bronze.central_bank_raw
    WHERE series_name = 'Vísitala neysluverðs'
""")

if df_silver.isEmpty():
    raise ValueError("[silver_central_bank] No rows returned - halting.")

df_silver.groupBy("metric").count().show()

In [ ]:
df_silver.createOrReplaceTempView("v_cb_silver")

if not spark.catalog.tableExists("silver.central_bank_indicators"):
    df_silver.write.format("delta").saveAsTable("silver.central_bank_indicators")
else:
    spark.sql("""
        MERGE INTO silver.central_bank_indicators AS target
        USING v_cb_silver AS source
        ON target.date = source.date AND target.metric = source.metric
        WHEN MATCHED THEN UPDATE SET
            target.series_name   = source.series_name,
            target.value         = source.value,
            target.processed_at  = source.processed_at
        WHEN NOT MATCHED THEN INSERT (date, metric, series_name, value, processed_at)
        VALUES (source.date, source.metric, source.series_name, source.value, source.processed_at)
    """)

print(f"Saved to silver.central_bank_indicators - {spark.table('silver.central_bank_indicators').count()} rows")